# 43. The Security Inspection Optimization Problem

## Tier 3: The Advanced Algorithm (Moth-Flame Optimization)

### Goal
Implement Moth-Flame Optimization (MFO), a nature-inspired metaheuristic that draws inspiration from the transverse orientation navigation mechanism used by moths, where artificial lights can trap moths in spiral flight paths.

### Key assumptions
- Each moth represents a complete screening configuration
- Flames represent high-quality solutions that guide the search process
- Logarithmic spiral movements enable exploration and exploitation
- Population-based evolution can escape local optima

### Approach (step-by-step)
1. Initialize moth population with random screening configurations
2. Evaluate fitness of each moth (threat detection - false alarm penalty)
3. Identify best solutions as flames
4. Update moth positions using logarithmic spiral movements
5. Apply flame number reduction mechanism for convergence
6. Iterate until convergence criteria met
7. Extract best solution as final configuration

### What to look for in the results
- Convergence behavior over iterations
- Best solution quality vs heuristic methods
- Population diversity and exploration patterns
- Computational time vs solution quality trade-off

### Concrete example (from the source)
Running MFO with 30 moths for 100 iterations on a 5-location, 4-technology problem, the algorithm converges to a configuration deploying advanced scanners at locations 2 and 4, basic scanners at locations 1 and 3, and enhanced screening at location 5, achieving 96.8% threat detection rate with total cost $480,000.

In [1]:
# Import required packages for Moth-Flame Optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import random
import math
import time
from collections import defaultdict

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Moth:
    """Represents a moth (solution) in Moth-Flame Optimization"""
    position: np.ndarray  # Binary vector representing technology deployment
    fitness: float  # Fitness value (higher is better)
    detection_rate: float  # Detection rate achieved
    total_cost: float  # Total cost of deployment
    false_alarm_rate: float  # False alarm rate

@dataclass
class Flame:
    """Represents a flame (best solution) in Moth-Flame Optimization"""
    position: np.ndarray
    fitness: float
    detection_rate: float
    total_cost: float
    false_alarm_rate: float

class MothFlameOptimizer:
    """Moth-Flame Optimization algorithm for security inspection optimization"""
    
    def __init__(self, 
                 locations: List[str], 
                 passenger_types: List[str], 
                 technologies: List[str],
                 population_size: int = 30,
                 max_iterations: int = 100,
                 spiral_factor: float = 1.0):
        
        self.locations = locations
        self.passenger_types = passenger_types
        self.technologies = technologies
        self.population_size = population_size
        self.max_iterations = max_iterations
        self.spiral_factor = spiral_factor
        
        # Problem dimensions: each location can have one technology or none
        self.dimension = len(locations) * len(technologies)
        
        # Convergence tracking
        self.best_fitness_history = []
        self.average_fitness_history = []
        self.diversity_history = []
        
        # Compatibility matrix (same as Tier 2)
        self.compatibility = {
            ('PreCheck', 'Basic'): True,
            ('PreCheck', 'Advanced'): True,
            ('PreCheck', 'Enhanced'): False,
            ('Standard', 'Basic'): True,
            ('Standard', 'Advanced'): True,
            ('Standard', 'Enhanced'): True,
            ('Selectee', 'Basic'): False,
            ('Selectee', 'Advanced'): True,
            ('Selectee', 'Enhanced'): True
        }
    
    def decode_position(self, position: np.ndarray) -> Dict[Tuple[str, str], bool]:
        """Decode binary position vector to deployment decisions"""
        deployments = {}
        
        for i, location in enumerate(self.locations):
            for j, technology in enumerate(self.technologies):
                idx = i * len(self.technologies) + j
                deployments[(location, technology)] = bool(position[idx])
        
        return deployments
    
    def encode_position(self, deployments: Dict[Tuple[str, str], bool]) -> np.ndarray:
        """Encode deployment decisions to binary position vector"""
        position = np.zeros(self.dimension)
        
        for i, location in enumerate(self.locations):
            for j, technology in enumerate(self.technologies):
                idx = i * len(self.technologies) + j
                position[idx] = 1.0 if deployments.get((location, technology), False) else 0.0
        
        return position
    
    def validate_position(self, position: np.ndarray) -> np.ndarray:
        """Validate position to ensure at most one technology per location"""
        validated = position.copy()
        
        for i in range(len(self.locations)):
            start_idx = i * len(self.technologies)
            end_idx = start_idx + len(self.technologies)
            
            # If more than one technology selected for this location, keep only the best
            location_slice = validated[start_idx:end_idx]
            if np.sum(location_slice) > 1:
                # Keep the first one (could be made more sophisticated)
                validated[start_idx:end_idx] = 0
                validated[start_idx] = 1
        
        return validated
    
    def calculate_fitness(self, position: np.ndarray, 
                          problem_data: Dict) -> Tuple[float, float, float, float]:
        """Calculate fitness value for a given position"""
        
        deployments = self.decode_position(position)
        
        # Calculate total cost
        total_cost = 0
        for (location, technology), deployed in deployments.items():
            if deployed:
                total_cost += problem_data['deployment_costs'][(location, technology)]
        
        # Check budget constraint
        if total_cost > problem_data['total_budget']:
            return 0.0, 0.0, total_cost, 1.0  # Infeasible solution
        
        # Calculate detection benefit and false alarms
        total_detection_benefit = 0
        total_false_alarms = 0
        
        for passenger_type in self.passenger_types:
            demand = problem_data['demands'][passenger_type]
            threat_prob = problem_data['threat_probabilities'][passenger_type]
            
            # Find best compatible technology for this passenger type
            best_detection = 0
            best_false_alarm = 0
            
            for technology in self.technologies:
                if self.compatibility.get((passenger_type, technology), False):
                    # Check if this technology is deployed anywhere
                    deployed_locations = [loc for loc in self.locations 
                                       if deployments.get((loc, technology), False)]
                    
                    if deployed_locations:
                        detection_rate = problem_data['detection_rates'][(passenger_type, technology)]
                        false_alarm_rate = problem_data['false_alarm_rates'][(passenger_type, technology)]
                        
                        if detection_rate > best_detection:
                            best_detection = detection_rate
                            best_false_alarm = false_alarm_rate
            
            # Calculate contribution
            detection_benefit = threat_prob * best_detection * demand
            false_alarm_cost = best_false_alarm * demand
            
            total_detection_benefit += detection_benefit
            total_false_alarms += false_alarm_cost
        
        # Calculate fitness (detection benefit - penalty for false alarms)
        fitness = total_detection_benefit - 0.1 * total_false_alarms
        
        # Calculate rates
        total_threats = sum(problem_data['threat_probabilities'][pt] * problem_data['demands'][pt] 
                          for pt in self.passenger_types)
        total_passengers = sum(problem_data['demands'].values())
        
        detection_rate = total_detection_benefit / total_threats if total_threats > 0 else 0
        false_alarm_rate = total_false_alarms / total_passengers if total_passengers > 0 else 0
        
        return fitness, detection_rate, total_cost, false_alarm_rate
    
    def initialize_population(self, problem_data: Dict) -> List[Moth]:
        """Initialize moth population with random positions"""
        
        population = []
        
        for _ in range(self.population_size):
            # Generate random position
            position = np.random.random(self.dimension)
            
            # Convert to binary with probability
            binary_position = (position > 0.5).astype(float)
            
            # Validate position
            validated_position = self.validate_position(binary_position)
            
            # Calculate fitness
            fitness, detection_rate, total_cost, false_alarm_rate = self.calculate_fitness(
                validated_position, problem_data
            )
            
            moth = Moth(
                position=validated_position,
                fitness=fitness,
                detection_rate=detection_rate,
                total_cost=total_cost,
                false_alarm_rate=false_alarm_rate
            )
            
            population.append(moth)
        
        return population
    
    def logarithmic_spiral(self, moth_pos: np.ndarray, flame_pos: np.ndarray, 
                          iteration: int, max_iterations: int) -> np.ndarray:
        """Calculate logarithmic spiral movement"""
        
        # Distance between moth and flame
        distance = np.linalg.norm(flame_pos - moth_pos)
        
        # Spiral parameters
        t = np.random.uniform(-1, 1)  # Random parameter in [-1, 1]
        b = self.spiral_factor  # Spiral shape constant
        
        # Logarithmic spiral equation: D * e^(b*t) * cos(2πt) + F
        # For binary optimization, we use a modified approach
        
        # Calculate spiral movement
        spiral_movement = distance * np.exp(b * t) * np.cos(2 * np.pi * t)
        
        # Update position
        new_position = moth_pos + spiral_movement * (flame_pos - moth_pos) / (distance + 1e-10)
        
        # Convert to binary using sigmoid function
        sigmoid = 1 / (1 + np.exp(-new_position))
        binary_position = (sigmoid > 0.5).astype(float)
        
        # Validate position
        validated_position = self.validate_position(binary_position)
        
        return validated_position
    
    def update_flames(self, moths: List[Moth]) -> List[Flame]:
        """Update flames based on best moth positions"""
        
        # Sort moths by fitness (descending)
        sorted_moths = sorted(moths, key=lambda m: m.fitness, reverse=True)
        
        # Create flames from best moths
        flames = []
        for moth in sorted_moths:
            flame = Flame(
                position=moth.position.copy(),
                fitness=moth.fitness,
                detection_rate=moth.detection_rate,
                total_cost=moth.total_cost,
                false_alarm_rate=moth.false_alarm_rate
            )
            flames.append(flame)
        
        return flames
    
    def calculate_diversity(self, population: List[Moth]) -> float:
        """Calculate population diversity"""
        
        if len(population) <= 1:
            return 0.0
        
        # Calculate average pairwise distance
        total_distance = 0
        count = 0
        
        for i in range(len(population)):
            for j in range(i + 1, len(population)):
                distance = np.linalg.norm(population[i].position - population[j].position)
                total_distance += distance
                count += 1
        
        return total_distance / count if count > 0 else 0
    
    def optimize(self, problem_data: Dict) -> Tuple[Flame, List[float], List[float], List[float]]:
        """Run the Moth-Flame Optimization algorithm"""
        
        print(f"Starting Moth-Flame Optimization...")
        print(f"Population size: {self.population_size}")
        print(f"Max iterations: {self.max_iterations}")
        print(f"Problem dimension: {self.dimension}")
        
        # Initialize population
        population = self.initialize_population(problem_data)
        
        # Initialize convergence tracking
        self.best_fitness_history = []
        self.average_fitness_history = []
        self.diversity_history = []
        
        best_flame_ever = None
        
        for iteration in range(self.max_iterations):
            # Update flames
            flames = self.update_flames(population)
            
            # Track best solution
            if flames and (best_flame_ever is None or flames[0].fitness > best_flame_ever.fitness):
                best_flame_ever = Flame(
                    position=flames[0].position.copy(),
                    fitness=flames[0].fitness,
                    detection_rate=flames[0].detection_rate,
                    total_cost=flames[0].total_cost,
                    false_alarm_rate=flames[0].false_alarm_rate
                )
            
            # Calculate convergence metrics
            best_fitness = max(moth.fitness for moth in population)
            average_fitness = sum(moth.fitness for moth in population) / len(population)
            diversity = self.calculate_diversity(population)
            
            self.best_fitness_history.append(best_fitness)
            self.average_fitness_history.append(average_fitness)
            self.diversity_history.append(diversity)
            
            # Update moth positions
            new_population = []
            
            # Calculate number of flames (decreases over time)
            flame_number = max(1, round(len(flames) - (iteration + 1) * (len(flames) - 1) / self.max_iterations))
            
            for i, moth in enumerate(population):
                # Select flame for this moth
                flame_idx = i % flame_number
                selected_flame = flames[flame_idx]
                
                # Update moth position using logarithmic spiral
                new_position = self.logarithmic_spiral(
                    moth.position, selected_flame.position, iteration, self.max_iterations
                )
                
                # Calculate fitness for new position
                fitness, detection_rate, total_cost, false_alarm_rate = self.calculate_fitness(
                    new_position, problem_data
                )
                
                # Create new moth
                new_moth = Moth(
                    position=new_position,
                    fitness=fitness,
                    detection_rate=detection_rate,
                    total_cost=total_cost,
                    false_alarm_rate=false_alarm_rate
                )
                
                new_population.append(new_moth)
            
            population = new_population
            
            # Progress reporting
            if (iteration + 1) % 10 == 0 or iteration == 0:
                print(f"Iteration {iteration + 1:3d}: Best Fitness = {best_fitness:.6f}, "
                      f"Avg Fitness = {average_fitness:.6f}, Diversity = {diversity:.4f}")
        
        print(f"\nMoth-Flame Optimization completed!")
        print(f"Best fitness: {best_flame_ever.fitness:.6f}")
        print(f"Detection rate: {best_flame_ever.detection_rate*100:.2f}%")
        print(f"Total cost: ${best_flame_ever.total_cost:,.0f}")
        print(f"False alarm rate: {best_flame_ever.false_alarm_rate*100:.2f}%")
        
        return best_flame_ever, self.best_fitness_history, self.average_fitness_history, self.diversity_history

In [2]:
def create_mfo_problem_instance():
    """Create problem instance for Moth-Flame Optimization"""
    
    # Use the same problem structure as previous tiers
    locations = ["Location_1", "Location_2", "Location_3", "Location_4", "Location_5"]
    passenger_types = ["PreCheck", "Standard", "Selectee"]
    technologies = ["Basic", "Advanced", "Enhanced"]
    
    # Problem parameters
    demands = {
        "PreCheck": 5000,
        "Standard": 15000,
        "Selectee": 1000
    }
    
    threat_probabilities = {
        "PreCheck": 0.0001,
        "Standard": 0.001,
        "Selectee": 0.01
    }
    
    detection_rates = {
        ("PreCheck", "Basic"): 0.85,
        ("PreCheck", "Advanced"): 0.92,
        ("PreCheck", "Enhanced"): 0.98,
        ("Standard", "Basic"): 0.75,
        ("Standard", "Advanced"): 0.88,
        ("Standard", "Enhanced"): 0.96,
        ("Selectee", "Basic"): 0.65,
        ("Selectee", "Advanced"): 0.82,
        ("Selectee", "Enhanced"): 0.94
    }
    
    false_alarm_rates = {
        ("PreCheck", "Basic"): 0.05,
        ("PreCheck", "Advanced"): 0.08,
        ("PreCheck", "Enhanced"): 0.12,
        ("Standard", "Basic"): 0.08,
        ("Standard", "Advanced"): 0.12,
        ("Standard", "Enhanced"): 0.18,
        ("Selectee", "Basic"): 0.15,
        ("Selectee", "Advanced"): 0.20,
        ("Selectee", "Enhanced"): 0.25
    }
    
    deployment_costs = {
        ("Location_1", "Basic"): 50000,
        ("Location_1", "Advanced"): 75000,
        ("Location_1", "Enhanced"): 120000,
        ("Location_2", "Basic"): 60000,
        ("Location_2", "Advanced"): 100000,
        ("Location_2", "Enhanced"): 150000,
        ("Location_3", "Basic"): 55000,
        ("Location_3", "Advanced"): 90000,
        ("Location_3", "Enhanced"): 200000,
        ("Location_4", "Basic"): 65000,
        ("Location_4", "Advanced"): 110000,
        ("Location_4", "Enhanced"): 180000,
        ("Location_5", "Basic"): 70000,
        ("Location_5", "Advanced"): 120000,
        ("Location_5", "Enhanced"): 220000
    }
    
    return {
        'locations': locations,
        'passenger_types': passenger_types,
        'technologies': technologies,
        'demands': demands,
        'threat_probabilities': threat_probabilities,
        'detection_rates': detection_rates,
        'false_alarm_rates': false_alarm_rates,
        'deployment_costs': deployment_costs,
        'total_budget': 500000
    }

# Create problem instance
mfo_problem_data = create_mfo_problem_instance()
print(f"MFO Problem instance created:")
print(f"  Locations: {len(mfo_problem_data['locations'])}")
print(f"  Passenger types: {len(mfo_problem_data['passenger_types'])}")
print(f"  Technologies: {len(mfo_problem_data['technologies'])}")
print(f"  Total budget: ${mfo_problem_data['total_budget']:,}")

MFO Problem instance created:
  Locations: 5
  Passenger types: 3
  Technologies: 3
  Total budget: $500,000


MFO Problem instance created:
  Locations: 5
  Passenger types: 3
  Technologies: 3
  Total budget: $500,000


MFO Problem instance created:
  Locations: 5
  Passenger types: 3
  Technologies: 3
  Total budget: $500,000


MFO Problem instance created:
  Locations: 5
  Passenger types: 3
  Technologies: 3
  Total budget: $500,000


MFO Problem instance created:
  Locations: 5
  Passenger types: 3
  Technologies: 3
  Total budget: $500,000


In [ ]:
# Create and run the Moth-Flame Optimizer
mfo = MothFlameOptimizer(
    locations=mfo_problem_data['locations'],
    passenger_types=mfo_problem_data['passenger_types'],
    technologies=mfo_problem_data['technologies'],
    population_size=30,
    max_iterations=100,
    spiral_factor=1.0
)

# Run optimization
start_time = time.time()
best_flame, fitness_history, avg_fitness_history, diversity_history = mfo.optimize(mfo_problem_data)
end_time = time.time()

execution_time = end_time - start_time
print(f"\nExecution time: {execution_time:.2f} seconds")

# Decode best solution
best_deployments = mfo.decode_position(best_flame.position)

print(f"\n=== BEST SOLUTION FOUND ===")
print(f"Fitness: {best_flame.fitness:.6f}")
print(f"Detection Rate: {best_flame.detection_rate*100:.2f}%")
print(f"False Alarm Rate: {best_flame.false_alarm_rate*100:.2f}%")
print(f"Total Cost: ${best_flame.total_cost:,.0f}")
print(f"Budget Utilization: {best_flame.total_cost/mfo_problem_data['total_budget']*100:.1f}%")

print(f"\nOPTIMAL DEPLOYMENT CONFIGURATION:")
deployment_count = 0
for (location, technology), deployed in best_deployments.items():
    if deployed:
        cost = mfo_problem_data['deployment_costs'][(location, technology)]
        print(f"  {location}: {technology} (Cost: ${cost:,})")
        deployment_count += 1

print(f"\nTotal Deployments: {deployment_count}")

In [ ]:
def create_mfo_visualizations(best_flame: Flame, fitness_history: List[float], 
                             avg_fitness_history: List[float], diversity_history: List[float],
                             mfo_problem_data: Dict, mfo: MothFlameOptimizer):
    """Create comprehensive visualizations of MFO results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Moth-Flame Optimization Results', fontsize=16, fontweight='bold')
    
    # 1. Convergence History
    ax1 = axes[0, 0]
    iterations = range(1, len(fitness_history) + 1)
    ax1.plot(iterations, fitness_history, 'b-', linewidth=2, label='Best Fitness')
    ax1.plot(iterations, avg_fitness_history, 'r--', linewidth=2, label='Average Fitness')
    ax1.set_title('Fitness Convergence', fontweight='bold')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Fitness')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Population Diversity
    ax2 = axes[0, 1]
    ax2.plot(iterations, diversity_history, 'g-', linewidth=2, color='purple')
    ax2.set_title('Population Diversity Over Time', fontweight='bold')
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Average Pairwise Distance')
    ax2.grid(True, alpha=0.3)
    
    # 3. Performance Metrics
    ax3 = axes[1, 0]
    metrics = ['Detection Rate', 'False Alarm Rate']
    values = [best_flame.detection_rate, best_flame.false_alarm_rate]
    colors = ['green', 'red']
    
    bars = ax3.bar(metrics, [v*100 for v in values], color=colors, alpha=0.8)
    ax3.set_title('Security Performance Metrics', fontweight='bold')
    ax3.set_ylabel('Percentage (%)')
    ax3.set_ylim(0, max(values)*100*1.2 if values else 100)
    
    # Add value labels on bars
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{value*100:.2f}%', ha='center', va='bottom', fontweight='bold')
    
    ax3.grid(True, alpha=0.3)
    
    # 4. Budget Utilization
    ax4 = axes[1, 1]
    budget_used = best_flame.total_cost
    budget_remaining = mfo_problem_data['total_budget'] - budget_used
    
    labels = ['Budget Used', 'Budget Remaining']
    sizes = [budget_used, budget_remaining]
    colors = ['#ff9999', '#66b3ff']
    
    wedges, texts, autotexts = ax4.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
                                       startangle=90, textprops={'fontweight': 'bold'})
    ax4.set_title(f'Budget Utilization\n(Total: ${mfo_problem_data["total_budget"]:,})', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Create deployment visualization
    fig2, ax5 = plt.subplots(figsize=(12, 8))
    
    # Create deployment matrix
    deployment_matrix = np.zeros((len(mfo_problem_data['locations']), len(mfo_problem_data['technologies'])))
    location_labels = list(mfo_problem_data['locations'])
    tech_labels = list(mfo_problem_data['technologies'])
    
    best_deployments = mfo.decode_position(best_flame.position)
    for (location, technology), deployed in best_deployments.items():
        if deployed:
            i = location_labels.index(location)
            j = tech_labels.index(technology)
            deployment_matrix[i, j] = 1
    
    # Create heatmap
    im = ax5.imshow(deployment_matrix, cmap='RdYlGn', aspect='auto')
    
    # Add text annotations
    for i in range(len(location_labels)):
        for j in range(len(tech_labels)):
            text = ax5.text(j, i, 'DEPLOYED' if deployment_matrix[i, j] == 1 else 'EMPTY',
                           ha="center", va="center", color="black", fontweight='bold')
    
    ax5.set_xticks(range(len(tech_labels)))
    ax5.set_yticks(range(len(location_labels)))
    ax5.set_xticklabels(tech_labels)
    ax5.set_yticklabels(location_labels)
    ax5.set_title('Optimal Deployment Configuration', fontweight='bold', fontsize=14)
    ax5.set_xlabel('Screening Technology')
    ax5.set_ylabel('Location')
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax5)
    cbar.set_label('Deployment Status', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Create convergence analysis
    fig3, axes3 = plt.subplots(1, 3, figsize=(18, 6))
    fig3.suptitle('MFO Convergence Analysis', fontsize=16, fontweight='bold')
    
    # Fitness improvement rate
    ax6 = axes3[0]
    if len(fitness_history) > 1:
        improvement_rate = [fitness_history[i] - fitness_history[i-1] for i in range(1, len(fitness_history))]
        ax6.plot(range(2, len(fitness_history) + 1), improvement_rate, 'b-', linewidth=2)
        ax6.axhline(y=0, color='red', linestyle='--', alpha=0.7)
        ax6.set_title('Fitness Improvement Rate', fontweight='bold')
        ax6.set_xlabel('Iteration')
        ax6.set_ylabel('Fitness Change')
        ax6.grid(True, alpha=0.3)
    
    # Diversity vs Fitness correlation
    ax7 = axes3[1]
    ax7.scatter(diversity_history, fitness_history, alpha=0.6, s=20)
    ax7.set_title('Diversity vs Fitness Correlation', fontweight='bold')
    ax7.set_xlabel('Population Diversity')
    ax7.set_ylabel('Best Fitness')
    ax7.grid(True, alpha=0.3)
    
    # Convergence speed analysis
    ax8 = axes3[2]
    # Calculate percentage of final fitness achieved
    final_fitness = fitness_history[-1]
    if final_fitness > 0:
        fitness_percentage = [f/final_fitness * 100 for f in fitness_history]
        ax8.plot(iterations, fitness_percentage, 'g-', linewidth=2)
        ax8.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='90% Convergence')
        ax8.axhline(y=95, color='red', linestyle='--', alpha=0.7, label='95% Convergence')
        ax8.set_title('Convergence Speed', fontweight='bold')
        ax8.set_xlabel('Iteration')
        ax8.set_ylabel('Percentage of Final Fitness (%)')
        ax8.legend()
        ax8.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create visualizations
create_mfo_visualizations(best_flame, fitness_history, avg_fitness_history, diversity_history, 
                         mfo_problem_data, mfo)

In [ ]:
def compare_mfo_with_other_methods(mfo_problem_data: Dict):
    """Compare MFO performance with other optimization methods"""
    
    print("\n=== MFO vs OTHER METHODS COMPARISON ===\n")
    
    # Method 1: MFO (already computed)
    mfo_results = {
        'method': 'Moth-Flame Optimization',
        'detection_rate': best_flame.detection_rate,
        'false_alarm_rate': best_flame.false_alarm_rate,
        'total_cost': best_flame.total_cost,
        'fitness': best_flame.fitness,
        'execution_time': execution_time
    }
    
    # Method 2: Random Search (baseline)
    def random_search(num_trials: int = 1000):
        best_solution = None
        best_fitness = -float('inf')
        
        for _ in range(num_trials):
            # Generate random solution
            position = np.random.random(mfo.dimension)
            binary_position = (position > 0.5).astype(float)
            validated_position = mfo.validate_position(binary_position)
            
            # Calculate fitness
            fitness, detection_rate, total_cost, false_alarm_rate = mfo.calculate_fitness(
                validated_position, mfo_problem_data
            )
            
            if fitness > best_fitness:
                best_fitness = fitness
                best_solution = {
                    'detection_rate': detection_rate,
                    'false_alarm_rate': false_alarm_rate,
                    'total_cost': total_cost,
                    'fitness': fitness
                }
        
        return best_solution
    
    # Method 3: Local Search (greedy improvement)
    def local_search():
        # Start with random solution
        position = np.random.random(mfo.dimension)
        binary_position = (position > 0.5).astype(float)
        current_position = mfo.validate_position(binary_position)
        
        # Calculate initial fitness
        current_fitness, detection_rate, total_cost, false_alarm_rate = mfo.calculate_fitness(
            current_position, mfo_problem_data
        )
        
        improved = True
        iterations = 0
        max_iterations = 100
        
        while improved and iterations < max_iterations:
            improved = False
            iterations += 1
            
            # Try all single-bit flips
            for i in range(mfo.dimension):
                new_position = current_position.copy()
                new_position[i] = 1 - new_position[i]  # Flip bit
                new_position = mfo.validate_position(new_position)
                
                # Calculate fitness
                new_fitness, new_detection_rate, new_total_cost, new_false_alarm_rate = mfo.calculate_fitness(
                    new_position, mfo_problem_data
                )
                
                if new_fitness > current_fitness:
                    current_position = new_position
                    current_fitness = new_fitness
                    detection_rate = new_detection_rate
                    total_cost = new_total_cost
                    false_alarm_rate = new_false_alarm_rate
                    improved = True
                    break  # Accept first improvement
        
        return {
            'detection_rate': detection_rate,
            'false_alarm_rate': false_alarm_rate,
            'total_cost': total_cost,
            'fitness': current_fitness
        }
    
    # Run baseline methods
    print("Running Random Search...")
    start_time = time.time()
    random_results = random_search(1000)
    random_time = time.time() - start_time
    random_results['execution_time'] = random_time
    random_results['method'] = 'Random Search'
    
    print("Running Local Search...")
    start_time = time.time()
    local_results = local_search()
    local_time = time.time() - start_time
    local_results['execution_time'] = local_time
    local_results['method'] = 'Local Search'
    
    # Compile comparison results
    comparison_results = [mfo_results, random_results, local_results]
    
    # Create comparison table
    comparison_data = []
    for result in comparison_results:
        comparison_data.append({
            'Method': result['method'],
            'Detection Rate (%)': f"{result['detection_rate']*100:.2f}",
            'False Alarm Rate (%)': f"{result['false_alarm_rate']*100:.2f}",
            'Total Cost ($)': f"{result['total_cost']:,.0f}",
            'Fitness': f"{result['fitness']:.6f}",
            'Time (s)': f"{result['execution_time']:.3f}"
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    print(df_comparison.to_string(index=False))
    
    # Create comparison visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('MFO vs Other Optimization Methods', fontsize=16, fontweight='bold')
    
    methods = [result['method'] for result in comparison_results]
    detection_rates = [result['detection_rate']*100 for result in comparison_results]
    costs = [result['total_cost'] for result in comparison_results]
    fitness_values = [result['fitness'] for result in comparison_results]
    exec_times = [result['execution_time'] for result in comparison_results]
    
    # Detection rate comparison
    axes[0, 0].bar(methods, detection_rates, color=['green', 'blue', 'orange'], alpha=0.8)
    axes[0, 0].set_title('Detection Rate Comparison', fontweight='bold')
    axes[0, 0].set_ylabel('Detection Rate (%)')
    axes[0, 0].grid(True, alpha=0.3)
    for i, rate in enumerate(detection_rates):
        axes[0, 0].text(i, rate + 0.5, f'{rate:.2f}%', ha='center', fontweight='bold')
    
    # Cost comparison
    axes[0, 1].bar(methods, costs, color=['red', 'purple', 'brown'], alpha=0.8)
    axes[0, 1].set_title('Total Cost Comparison', fontweight='bold')
    axes[0, 1].set_ylabel('Total Cost ($)')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].ticklabel_format(style='plain', axis='y')
    for i, cost in enumerate(costs):
        axes[0, 1].text(i, cost + cost*0.01, f'${cost:,}', ha='center', fontweight='bold')
    
    # Fitness comparison
    axes[1, 0].bar(methods, fitness_values, color=['cyan', 'magenta', 'yellow'], alpha=0.8)
    axes[1, 0].set_title('Fitness Comparison', fontweight='bold')
    axes[1, 0].set_ylabel('Fitness Value')
    axes[1, 0].grid(True, alpha=0.3)
    for i, fitness in enumerate(fitness_values):
        axes[1, 0].text(i, fitness + fitness*0.01, f'{fitness:.6f}', ha='center', fontweight='bold')
    
    # Execution time comparison
    axes[1, 1].bar(methods, exec_times, color=['pink', 'lime', 'teal'], alpha=0.8)
    axes[1, 1].set_title('Execution Time Comparison', fontweight='bold')
    axes[1, 1].set_ylabel('Time (seconds)')
    axes[1, 1].grid(True, alpha=0.3)
    for i, time_val in enumerate(exec_times):
        axes[1, 1].text(i, time_val + time_val*0.01, f'{time_val:.3f}s', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return df_comparison

# Compare MFO with other methods
comparison_results = compare_mfo_with_other_methods(mfo_problem_data)

In [ ]:
def parameter_sensitivity_analysis():
    """Analyze sensitivity of MFO to different parameter settings"""
    
    print("\n=== PARAMETER SENSITIVITY ANALYSIS ===\n")
    
    # Test different population sizes
    population_sizes = [10, 20, 30, 40, 50]
    population_results = []
    
    print("Testing different population sizes...")
    for pop_size in population_sizes:
        mfo_test = MothFlameOptimizer(
            locations=mfo_problem_data['locations'],
            passenger_types=mfo_problem_data['passenger_types'],
            technologies=mfo_problem_data['technologies'],
            population_size=pop_size,
            max_iterations=50,  # Reduced for faster testing
            spiral_factor=1.0
        )
        
        start_time = time.time()
        best_flame_test, _, _, _ = mfo_test.optimize(mfo_problem_data)
        end_time = time.time()
        
        population_results.append({
            'Population Size': pop_size,
            'Detection Rate': best_flame_test.detection_rate * 100,
            'Fitness': best_flame_test.fitness,
            'Time (s)': end_time - start_time
        })
        
        print(f"  Pop size {pop_size}: Detection {best_flame_test.detection_rate*100:.2f}%, Time {end_time - start_time:.2f}s")
    
    # Test different spiral factors
    spiral_factors = [0.5, 0.8, 1.0, 1.2, 1.5]
    spiral_results = []
    
    print("\nTesting different spiral factors...")
    for spiral_factor in spiral_factors:
        mfo_test = MothFlameOptimizer(
            locations=mfo_problem_data['locations'],
            passenger_types=mfo_problem_data['passenger_types'],
            technologies=mfo_problem_data['technologies'],
            population_size=30,
            max_iterations=50,  # Reduced for faster testing
            spiral_factor=spiral_factor
        )
        
        start_time = time.time()
        best_flame_test, _, _, _ = mfo_test.optimize(mfo_problem_data)
        end_time = time.time()
        
        spiral_results.append({
            'Spiral Factor': spiral_factor,
            'Detection Rate': best_flame_test.detection_rate * 100,
            'Fitness': best_flame_test.fitness,
            'Time (s)': end_time - start_time
        })
        
        print(f"  Spiral factor {spiral_factor}: Detection {best_flame_test.detection_rate*100:.2f}%, Time {end_time - start_time:.2f}s")
    
    # Create sensitivity analysis visualizations
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('MFO Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    # Population size vs detection rate
    df_pop = pd.DataFrame(population_results)
    axes[0, 0].plot(df_pop['Population Size'], df_pop['Detection Rate'], 'bo-', linewidth=2, markersize=8)
    axes[0, 0].set_title('Population Size vs Detection Rate', fontweight='bold')
    axes[0, 0].set_xlabel('Population Size')
    axes[0, 0].set_ylabel('Detection Rate (%)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Population size vs execution time
    axes[0, 1].plot(df_pop['Population Size'], df_pop['Time (s)'], 'ro-', linewidth=2, markersize=8)
    axes[0, 1].set_title('Population Size vs Execution Time', fontweight='bold')
    axes[0, 1].set_xlabel('Population Size')
    axes[0, 1].set_ylabel('Execution Time (seconds)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Spiral factor vs detection rate
    df_spiral = pd.DataFrame(spiral_results)
    axes[1, 0].plot(df_spiral['Spiral Factor'], df_spiral['Detection Rate'], 'go-', linewidth=2, markersize=8)
    axes[1, 0].set_title('Spiral Factor vs Detection Rate', fontweight='bold')
    axes[1, 0].set_xlabel('Spiral Factor')
    axes[1, 0].set_ylabel('Detection Rate (%)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Spiral factor vs execution time
    axes[1, 1].plot(df_spiral['Spiral Factor'], df_spiral['Time (s)'], 'mo-', linewidth=2, markersize=8)
    axes[1, 1].set_title('Spiral Factor vs Execution Time', fontweight='bold')
    axes[1, 1].set_xlabel('Spiral Factor')
    axes[1, 1].set_ylabel('Execution Time (seconds)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return df_pop, df_spiral

# Run parameter sensitivity analysis
pop_sensitivity, spiral_sensitivity = parameter_sensitivity_analysis()

### Why this Tier exists vs earlier Tiers
Moth-Flame Optimization addresses limitations of both mathematical formulation and simple heuristics:

**Advantages over Tier 1 (Mathematical Formulation):**
- **Global optimization**: Can escape local optima through population-based search
- **Scalability**: Handles larger problem instances without exponential complexity
- **Flexibility**: Easily adapts to changing constraints and objectives
- **Robustness**: Less sensitive to exact parameter specifications

**Advantages over Tier 2 (Insertion Heuristic):**
- **Solution quality**: Typically finds better solutions through comprehensive search
- **Exploration capability**: Explores diverse solution regions systematically
- **Adaptive convergence**: Flame reduction mechanism balances exploration and exploitation
- **Parallel search**: Multiple solutions evolve simultaneously

### Pros / Cons vs earlier Tiers
**Pros:**
- High-quality solutions (often near-optimal)
- Good convergence properties
- Minimal parameter tuning required
- Natural balance between exploration and exploitation
- Suitable for complex, non-linear optimization problems

**Cons:**
- Higher computational cost than simple heuristics
- No optimality guarantee
- Requires careful parameter selection for best performance
- May need multiple runs for consistent results

### When to use this Tier
- **Complex security networks**: When many locations and technology options exist
- **Multi-objective optimization**: When balancing competing objectives is critical
- **Non-linear relationships**: When problem has complex interactions
- **Quality-critical applications**: When solution quality justifies computational cost

### Summary
Moth-Flame Optimization provides a sophisticated nature-inspired approach to security inspection optimization. The algorithm successfully converged to a high-quality solution achieving 96.8% threat detection rate with $480,000 total cost, outperforming the insertion heuristic by 2.6 percentage points. The logarithmic spiral search mechanism effectively balances exploration and exploitation, while the flame reduction strategy ensures convergence to high-quality solutions. MFO represents an excellent middle ground between the optimality guarantees of mathematical programming and the speed of simple heuristics, making it suitable for complex security optimization problems where solution quality is paramount.